In [ ]:
import pandas as pd
import requests
import os
import time
import random
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# PART 1: DATA PREPARATION & BRAND FILTERING
# ==========================================
print("Step 1: Preparing and filtering laptop data...")

df = pd.read_csv('laptop.csv')

# Apply advanced extraction pattern
pattern = r'^(.+?)\s+(?:Laptops?|Ultrabook|Netbook|Notebook)?\s*\((.+?)\)\s*::'
df[['Model_Name', 'Specs']] = df['Name'].str.extract(pattern)
df['Model_Name'] = df['Model_Name'].str.strip()

# Target brands filter
target_brands = ['hp', 'asus', 'acer', 'lenovo', 'msi', 'gigabyte', 'dell', 'apple', 'razer']
df = df[df['Brand'].str.lower().isin(target_brands)]

# Sample 5 models per brand
sampled_df = df.dropna(subset=['Model_Name']).groupby('Brand', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 5), random_state=42)
).reset_index(drop=True)

print(f"✅ Selected {len(sampled_df)} models across {sampled_df['Brand'].nunique()} target brands.")


# ==========================================
# PART 2: STABLE BING SELENIUM SCRAPER
# ==========================================
print("\nStep 2: Initializing Stable Bing Scraper...")

SAVE_DIR = "bing_selenium_images"
TARGET_IMAGES = 50

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

VIEW_MODIFIERS = {
    "frontal": "keyboard open front view",
    "ports": "side ports view",
    "lid": "lid cover logo closed"
}

# --- STABLE SELENIUM SETUP ---
options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-notifications')

# Basic stealth flags to prevent Bing from showing popups
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# This automatically handles driver version matching so it NEVER crashes!
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("Browser ready. Starting loop...\n")

for index, row in sampled_df.iterrows():
    brand = row['Brand']
    model_name = row['Model_Name'] 
    
    print(f"--- [{index+1}/{len(sampled_df)}] Scraping {brand}: {model_name} ---")
    
    model_dir = os.path.join(SAVE_DIR, model_name.replace("/", "-").replace(":", ""))
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    for view_name, modifier in VIEW_MODIFIERS.items():
        query = f'"{model_name}" {modifier}'
        print(f"  -> Searching: {query}")
        
        # Navigate directly to Bing Images
        search_url = f"https://www.bing.com/images/search?q={query}"
        driver.get(search_url)
        time.sleep(random.uniform(2.0, 3.0)) 
        
        # SCROLLING TO LOAD IMAGES
        for _ in range(4):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(random.uniform(1.0, 2.0))
        
        try:
            # Locate Bing's image containers
            image_elements = driver.find_elements(By.CSS_SELECTOR, "a.iusc")
        except:
            print("     [!] Couldn't find image elements on Bing.")
            continue

        images_saved = 0
        
        for element in image_elements:
            if images_saved >= TARGET_IMAGES:
                break
                
            try:
                # The hidden Bing Magic: Get the high-res URL without clicking!
                m_data = element.get_attribute("m")
                if not m_data:
                    continue
                    
                json_data = json.loads(m_data)
                high_res_url = json_data.get("murl")
                
                if high_res_url:
                    # Download the image
                    img_bytes = requests.get(high_res_url, timeout=5).content
                    safe_filename = f"{model_name}_{view_name}_{images_saved}.jpg".replace("/", "-").replace(" ", "_")
                    file_path = os.path.join(model_dir, safe_filename)
                    
                    with open(file_path, 'wb') as handler:
                        handler.write(img_bytes)
                    
                    print(f"     [+] Downloaded image {images_saved+1}/{TARGET_IMAGES}")
                    images_saved += 1
                    
            except Exception:
                # Skip silently if download fails (e.g. 403 forbidden or broken link)
                continue

print("\n✅ All Bing scraping complete!")
driver.quit()

Step 1: Preparing and filtering laptop data...
✅ Selected 41 models across 9 target brands.

Step 2: Initializing Stable Bing Scraper...


C:\Users\marye\AppData\Local\Temp\ipykernel_9300\2652853670.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.dropna(subset=['Model_Name']).groupby('Brand', group_keys=False).apply(


Browser ready. Starting loop...

--- [1/41] Scraping ASUS: ASUS VivoBook Pro 15 M6500QC-HN751WS ---
  -> Searching: "ASUS VivoBook Pro 15 M6500QC-HN751WS" keyboard open front view
     [+] Downloaded image 1/50
     [+] Downloaded image 2/50
     [+] Downloaded image 3/50
     [+] Downloaded image 4/50
     [+] Downloaded image 5/50
     [+] Downloaded image 6/50
     [+] Downloaded image 7/50
     [+] Downloaded image 8/50
     [+] Downloaded image 9/50
     [+] Downloaded image 10/50
     [+] Downloaded image 11/50
     [+] Downloaded image 12/50
     [+] Downloaded image 13/50
     [+] Downloaded image 14/50
     [+] Downloaded image 15/50
     [+] Downloaded image 16/50
     [+] Downloaded image 17/50
     [+] Downloaded image 18/50
     [+] Downloaded image 19/50
     [+] Downloaded image 20/50
     [+] Downloaded image 21/50
     [+] Downloaded image 22/50
     [+] Downloaded image 23/50
     [+] Downloaded image 24/50
     [+] Downloaded image 25/50
     [+] Downloaded image 26/

In [ ]:
!pip install ImageHash Pillow

In [ ]:
import os
from PIL import Image
import imagehash

# The folder where your scraped images are saved
# Update this to "bing_selenium_images" or "bing_laptop_images" depending on which script you used
DATASET_DIR = "bing_selenium_images" 

def clean_image_dataset(dataset_path):
    print(f"Scanning directory: {dataset_path}...\n")
    
    # Dictionary to store our hashes: {hash_string: file_path}
    unique_hashes = {}
    
    duplicates_deleted = 0
    corrupt_deleted = 0
    total_scanned = 0

    # os.walk automatically goes through the main folder and all the model sub-folders
    for root, _, files in os.walk(dataset_path):
        for filename in files:
            # Only process image files
            if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                file_path = os.path.join(root, filename)
                total_scanned += 1
                
                try:
                    # Open the image using Pillow
                    with Image.open(file_path) as img:
                        # Calculate the Perceptual Hash (phash)
                        # phash is great at detecting duplicates even if one is slightly resized
                        img_hash = str(imagehash.phash(img))
                        
                        # If we have seen this exact visual hash before, it's a duplicate!
                        if img_hash in unique_hashes:
                            # Close the image so Windows lets us delete it
                            img.close() 
                            os.remove(file_path)
                            duplicates_deleted += 1
                            print(f" [🗑️ Deleted Duplicate] {filename}")
                        else:
                            # It's a new, unique image. Save its hash.
                            unique_hashes[img_hash] = file_path
                            
                except Exception as e:
                    # If Pillow can't open the file, it means the download was corrupted or broken.
                    try:
                        os.remove(file_path)
                        corrupt_deleted += 1
                        print(f" [⚠️ Deleted Corrupt/Broken File] {filename}")
                    except:
                        pass # Sometimes the OS locks the file temporarily

    print("\n" + "="*40)
    print("🧹 DATASET CLEANING COMPLETE")
    print("="*40)
    print(f"Total Images Scanned:   {total_scanned}")
    print(f"Unique Images Kept:     {len(unique_hashes)}")
    print(f"Duplicates Deleted:     {duplicates_deleted}")
    print(f"Corrupt Files Deleted:  {corrupt_deleted}")

# Run the cleaner
clean_image_dataset(DATASET_DIR)

Scanning directory: bing_selenium_images...

 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_3_A315-58_frontal_22.jpg
 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_3_A315-58_frontal_3.jpg
 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_3_A315-58_lid_31.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_lid_39.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_lid_44.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_0.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_21.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_27.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_35.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_36.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_40.jpg
 [🗑️ Deleted Duplicate] Acer_Aspire_3_A315-58_ports_46.jpg
 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_3_A315-58_ports_48.jpg
 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_3_A315-58_ports_9.jpg
 [⚠️ Deleted Corrupt/Broken File] Acer_Aspire_Lite_AL15-41_frontal_0.jpg
 [⚠️ Delete

c:\Users\marye\anaconda3\Lib\site-packages\PIL\Image.py:1000: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_M1_Z124J001KD_ports_19.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_M1_Z124J001KD_ports_21.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_M1_Z124J001KD_ports_22.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_11.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_13.jpg
 [🗑️ Deleted Duplicate] Apple_MacBook_Air_MRE82HN-A_frontal_14.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_15.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_18.jpg
 [🗑️ Deleted Duplicate] Apple_MacBook_Air_MRE82HN-A_frontal_2.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_24.jpg
 [🗑️ Deleted Duplicate] Apple_MacBook_Air_MRE82HN-A_frontal_28.jpg
 [⚠️ Deleted Corrupt/Broken File] Apple_MacBook_Air_MRE82HN-A_frontal_4.jpg
 [🗑️ Deleted Duplicate] Apple_MacBook_Air_MRE82HN-A_frontal_5.jpg
 [🗑️ Deleted Duplicate] Apple_MacBoo

In [ ]:
import pandas as pd
import requests
import os
import time
import random
import json
import re
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================
# PART 1: ADVANCED QUERY CLEANING & UTILITIES
# ==========================================
print("Step 1: Optimizing refrigerator search queries...")

# Load the complete dataset
df = pd.read_csv('refrigerator_complete.csv')

def encode_query_string(q):
    """Safely encodes spaces and special characters for a clean URL string"""
    return urllib.parse.quote_plus(q)

def optimize_fridge_query(name):
    """
    Cleans messy e-commerce titles into sharp, searchable model codes.
    Example: 'Réfrigérateur SAMSUNG RT31CG5000S9EL 330L NoFrost - Silver' 
    Becomes: 'RT31CG5000' or 'RT31'
    """
    # 1. Remove common e-commerce clutter words
    cleaned = re.sub(r'Réfrigérateur|NoFrost|DeFrost|LessFrost|LowFrost|Side-by-Side|Bottom Freezer|French Door|Double Door|Single Door|Multi-Portes|Door-in-Door|Instaview', '', name, flags=re.IGNORECASE)
    # 2. Remove colors and capacities
    cleaned = re.sub(r'\d+L|Silver|Inox|Blanc|Noir|Gris|Rouge|Verre', '', cleaned, flags=re.IGNORECASE)
    # 3. Remove text after dashes or colons
    cleaned = re.split(r'-|:', cleaned)[0]
    
    # 4. Extract the core alphanumeric model token (usually things like RT31CG..., KGN56..., GN-C422...)
    tokens = cleaned.split()
    core_model = ""
    for token in tokens:
        if any(c.isdigit() for c in token) and len(token) >= 3:
            core_model = token
            break
            
    if not core_model and len(tokens) > 1:
        core_model = " ".join(tokens[1:3])
    elif not core_model:
        core_model = tokens[0]
        
    return core_model.strip()

# Create clean, optimized search tokens
df['Clean_SKU'] = df['Name'].apply(optimize_fridge_query)
df['Folder_Key'] = df['Clean_SKU'].str.replace("/", "-").str.replace(" ", "_")

print("Optimized search terms sample:")
print(df[['Brand', 'Clean_SKU']].head())


# ==========================================
# PART 2: THE REFINED SCRAPER
# ==========================================
SAVE_DIR = "refrigerator_dataset_images"
TARGET_IMAGES_PER_VIEW = 50

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# Simplified, high-probability e-commerce keywords
VIEW_MODIFIERS = {
    "front_closed": "refrigerator",          
    "front_open": "refrigerator open inside", 
    "side": "refrigerator angled profile"     
}

options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    for index, row in df.iterrows():
        brand = row['Brand']
        sku = row['Clean_SKU']
        folder_name = row['Folder_Key']
        
        print(f"\n--- [{index+1}/{len(df)}] SCRAPING: {brand} {sku} ---")
        
        model_path = os.path.join(SAVE_DIR, folder_name)
        if not os.path.exists(model_path):
            os.makedirs(model_path)

        for view_name, modifier in VIEW_MODIFIERS.items():
            query = f"{brand} {sku} {modifier}"
            print(f"  -> Query: {query}")
            
            # encode_query_string is now properly defined above!
            driver.get(f"https://www.bing.com/images/search?q={encode_query_string(query)}")
            time.sleep(random.uniform(1.5, 2.5)) 

            for _ in range(3):
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
            
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, "a.iusc")
                images_found = 0
                
                for el in elements:
                    if images_found >= TARGET_IMAGES_PER_VIEW:
                        break
                    
                    m_data = el.get_attribute("m")
                    if not m_data: continue
                    
                    img_url = json.loads(m_data).get("murl")
                    
                    if img_url:
                        try:
                            r = requests.get(img_url, timeout=4)
                            if r.status_code == 200:
                                filename = f"{folder_name}_{view_name}_{images_found}.jpg"
                                with open(os.path.join(model_path, filename), 'wb') as f:
                                    f.write(r.content)
                                images_found += 1
                        except:
                            continue
                
                print(f"     [+] Successfully saved {images_found} images")
            except Exception as e:
                print(f"     [!] Skipping view due to error: {e}")

finally:
    driver.quit()
    print("\n✅ Scraping finished with optimized queries.")

Step 1: Optimizing refrigerator search queries...
Optimized search terms sample:
     Brand       Clean_SKU
0  SAMSUNG  RT31CG5000S9EL
1  SAMSUNG  RT35CG5000S9EL
2  SAMSUNG  RT42CB66448AEL
3  SAMSUNG     RT40K5012S8
4  SAMSUNG     RT40K5012S8

--- [1/45] SCRAPING: SAMSUNG RT31CG5000S9EL ---
  -> Query: SAMSUNG RT31CG5000S9EL refrigerator
     [+] Successfully saved 50 images
  -> Query: SAMSUNG RT31CG5000S9EL refrigerator open inside
     [+] Successfully saved 50 images
  -> Query: SAMSUNG RT31CG5000S9EL refrigerator angled profile
     [+] Successfully saved 50 images

--- [2/45] SCRAPING: SAMSUNG RT35CG5000S9EL ---
  -> Query: SAMSUNG RT35CG5000S9EL refrigerator
     [+] Successfully saved 29 images
  -> Query: SAMSUNG RT35CG5000S9EL refrigerator open inside
     [+] Successfully saved 50 images
  -> Query: SAMSUNG RT35CG5000S9EL refrigerator angled profile
     [+] Successfully saved 50 images

--- [3/45] SCRAPING: SAMSUNG RT42CB66448AEL ---
  -> Query: SAMSUNG RT42CB66448AEL refri

In [ ]:
import csv, os, re, time, random
from urllib.parse import urlparse
import requests

# ---- settings: edit these ----
CSV_PATH = r"C:\Users\marye\Desktop\PFE MASTER\core_ai_setup\exploration\desktop_dataset.csv"
OUT_DIR  = "images"                 # where images go
URL_COL  = "Image_Path"             # column holding the URL
# -------------------------------

MIN_DELAY, MAX_DELAY, MAX_RETRIES, TIMEOUT = 1.5, 3.0, 3, 30
HEADERS = {
    "User-Agent": "Mozilla/5.0 (image archival script; contact: you@example.com)",
    "Accept": "image/avif,image/webp,image/*,*/*;q=0.8",
}

def slugify(brand, model):
    s = re.sub(r"[^a-z0-9]+", "-", f"{brand}_{model}".lower()).strip("-")
    return s[:120]

def download_one(session, url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return "skip (already exists)"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=TIMEOUT, stream=True)
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                return f"error (network): {e}"
            time.sleep(2 ** attempt); continue
        if resp.status_code == 200:
            tmp = dest + ".part"
            with open(tmp, "wb") as fh:
                for chunk in resp.iter_content(8192):
                    fh.write(chunk)
            os.replace(tmp, dest)
            return f"ok ({os.path.getsize(dest)} bytes)"
        if resp.status_code in (403, 429, 503):
            ra = resp.headers.get("Retry-After")
            wait = int(ra) if (ra and ra.isdigit()) else 2 ** attempt * 5
            if attempt == MAX_RETRIES:
                return f"blocked (HTTP {resp.status_code}) — server declined; not bypassing"
            time.sleep(wait); continue
        return f"error (HTTP {resp.status_code})"
    return "error (exhausted retries)"

os.makedirs(OUT_DIR, exist_ok=True)
with open(CSV_PATH, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

total = len(rows)
print(f"Found {total} rows in {CSV_PATH}\n")
session = requests.Session()
counts = {"ok": 0, "skip": 0, "error": 0, "blocked": 0}

for i, row in enumerate(rows, 1):
    url = (row.get(URL_COL) or "").strip()
    if not url:
        print(f"[{i}/{total}] (no URL) — skipping"); counts["error"] += 1; continue
    ext = os.path.splitext(urlparse(url).path)[1] or ".jpg"
    dest = os.path.join(OUT_DIR, slugify(row.get("Brand",""), row.get("Model_Name","")) + ext)
    status = download_one(session, url, dest)
    print(f"[{i}/{total}] {os.path.basename(dest)}: {status}")
    key = ("ok" if status.startswith("ok") else
           "skip" if status.startswith("skip") else
           "blocked" if status.startswith("blocked") else "error")
    counts[key] += 1
    if not status.startswith("skip"):
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

print("\n--- summary ---")
for k in ("ok","skip","blocked","error"):
    print(f"{k}: {counts[k]}")

Found 71 rows in C:\Users\marye\Desktop\PFE MASTER\core_ai_setup\exploration\desktop_dataset.csv

[1/71] (no URL) — skipping
[2/71] (no URL) — skipping
[3/71] (no URL) — skipping
[4/71] (no URL) — skipping
[5/71] (no URL) — skipping
[6/71] (no URL) — skipping
[7/71] (no URL) — skipping
[8/71] (no URL) — skipping
[9/71] (no URL) — skipping
[10/71] (no URL) — skipping
[11/71] (no URL) — skipping
[12/71] (no URL) — skipping
[13/71] (no URL) — skipping
[14/71] (no URL) — skipping
[15/71] (no URL) — skipping
[16/71] (no URL) — skipping
[17/71] (no URL) — skipping
[18/71] (no URL) — skipping
[19/71] (no URL) — skipping
[20/71] (no URL) — skipping
[21/71] (no URL) — skipping
[22/71] (no URL) — skipping
[23/71] (no URL) — skipping
[24/71] (no URL) — skipping
[25/71] (no URL) — skipping
[26/71] (no URL) — skipping
[27/71] (no URL) — skipping
[28/71] (no URL) — skipping
[29/71] (no URL) — skipping
[30/71] (no URL) — skipping
[31/71] (no URL) — skipping
[32/71] (no URL) — skipping
[33/71] (no URL

In [4]:
import os
import re
import time
import random
import csv
import json
from io import BytesIO
from pathlib import Path
from urllib.parse import quote

import requests
from PIL import Image
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH = Path(r"C:\Users\marye\Desktop\PFEMASTER\core_ai_setup\exploration\air_conditioner_dataset.csv")
OUT_DIR = Path(r"C:\Users\marye\Desktop\PFEMASTER\core_ai_setup\exploration\air_conditioner_dataset")
IMAGES_PER_QUERY = 8
MIN_IMAGE_SIZE = (300, 300)
MAX_MODELS = None  # set to 2 or 3 to test quickly

# ── Helpers ───────────────────────────────────────────────────────────────────

def safe_folder(text, maxlen=60):
    return re.sub(r"[^\w\-]", "_", str(text).strip())[:maxlen]


def build_query_variants(brand, model):
    brand = str(brand or "").strip()
    model = str(model or "").strip()
    base = f'"{brand} {model}"'.strip()
    if not base:
        return []
    return [
        f"{base} split air conditioner indoor unit front view",
        f"{base} air conditioner indoor unit product shot",
        f"{base} indoor unit white background",
        f"{base} air conditioner catalog photo",
    ]


def make_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1400,900")
    options.add_argument("--lang=fr-FR")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_bing_image_urls(driver, query, max_images=10):
    encoded = quote(query)
    driver.get(f"https://www.bing.com/images/search?q={encoded}&form=HDRSC2&first=1&qft=+filterui:imagesize-large")
    time.sleep(random.uniform(2.0, 3.5))

    for _ in range(4):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(random.uniform(0.8, 1.5))

    urls = []
    seen = set()

    for anchor in driver.find_elements(By.CSS_SELECTOR, "a.iusc"):
        m_data = anchor.get_attribute("m")
        if not m_data:
            continue
        try:
            payload = json.loads(m_data)
        except Exception:
            continue
        img_url = payload.get("murl") or payload.get("turl")
        if img_url and img_url not in seen and img_url.startswith("http"):
            seen.add(img_url)
            urls.append(img_url)
        if len(urls) >= max_images:
            break

    if len(urls) < max_images:
        for img in driver.find_elements(By.CSS_SELECTOR, "img.mimg"):
            src = img.get_attribute("src") or img.get_attribute("data-src") or ""
            if src and src.startswith("http") and src not in seen and "bing.com" not in src and "microsoft" not in src:
                seen.add(src)
                urls.append(src)
            if len(urls) >= max_images:
                break

    return urls


def download_image(url, dest: Path):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "image/avif,image/webp,image/*,*/*;q=0.8",
        "Referer": "https://www.bing.com/",
    }
    try:
        r = requests.get(url, headers=headers, timeout=12, stream=True, allow_redirects=True)
        r.raise_for_status()
        if "image" not in r.headers.get("Content-Type", ""):
            return False
        data = r.content
        if len(data) < 10000:
            return False
        ext = ".jpg"
        with open(dest.with_suffix(ext), "wb") as f:
            f.write(data)
        return True
    except Exception:
        return False


def load_rows(path: Path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def save_rows(path: Path, rows):
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


# ── Main scraping loop ────────────────────────────────────────────────────────
rows = load_rows(CSV_PATH)
if MAX_MODELS is not None:
    rows = rows[:MAX_MODELS]

print(f"Step 1: Loaded {len(rows)} air conditioner models from CSV.\n")
print("Step 2: Initializing Chrome Driver...")

driver = make_driver()
print("Browser ready. Starting loop...\n")

updated_paths = []

try:
    for idx, row in enumerate(rows, 1):
        brand = str(row.get("Brand", "")).strip()
        model = str(row.get("Model_Name") or row.get("Model") or "").strip()
        folder = OUT_DIR / f"{safe_folder(brand)}_{safe_folder(model)}"
        folder.mkdir(parents=True, exist_ok=True)

        print(f"--- [{idx}/{len(rows)}] Scraping {brand}: {model} ---")

        img_counter = 1
        for query in build_query_variants(brand, model):
            print(f"  -> Searching: {query}")
            try:
                urls = get_bing_image_urls(driver, query, max_images=IMAGES_PER_QUERY)
                downloaded = 0
                for url in urls:
                    if downloaded >= IMAGES_PER_QUERY:
                        break
                    dest = folder / f"img_{img_counter:03d}"
                    if download_image(url, dest):
                        downloaded += 1
                        img_counter += 1
                        print(f"     [+] Downloaded image {downloaded}/{IMAGES_PER_QUERY}")
                    time.sleep(random.uniform(0.2, 0.6))
            except Exception as e:
                print(f"     [!] Skipping query due to error: {e}")

            time.sleep(random.uniform(1.0, 2.0))

        updated_paths.append(str(folder))
        row["Image_Path"] = str(folder)
        print()

finally:
    driver.quit()
    print("\n✅ Scraping finished.")

save_rows(CSV_PATH, rows)
print(f"✅ CSV updated with Image_Path → {CSV_PATH}")

print("\n--- summary ---")
total_imgs = sum(len(list(Path(p).glob("*.jpg"))) for p in updated_paths if Path(p).exists())
print(f"Models scraped : {len(rows)}")
print(f"Total images   : {total_imgs}")
print(f"Output folder  : {OUT_DIR}")

Step 1: Loaded 25 air conditioner models from CSV.

Step 2: Initializing Chrome Driver...
Browser ready. Starting loop...

--- [1/25] Scraping CARRIER: Carrier 53KHCT-09GE ---
  -> Searching: "CARRIER Carrier 53KHCT-09GE" split air conditioner indoor unit front view
  -> Searching: "CARRIER Carrier 53KHCT-09GE" air conditioner indoor unit product shot
  -> Searching: "CARRIER Carrier 53KHCT-09GE" indoor unit white background
  -> Searching: "CARRIER Carrier 53KHCT-09GE" air conditioner catalog photo

--- [2/25] Scraping CARRIER: Carrier 53KHCT-12GE ---
  -> Searching: "CARRIER Carrier 53KHCT-12GE" split air conditioner indoor unit front view
  -> Searching: "CARRIER Carrier 53KHCT-12GE" air conditioner indoor unit product shot
  -> Searching: "CARRIER Carrier 53KHCT-12GE" indoor unit white background
  -> Searching: "CARRIER Carrier 53KHCT-12GE" air conditioner catalog photo

--- [3/25] Scraping CARRIER: Carrier 53KHCT-18GE ---
  -> Searching: "CARRIER Carrier 53KHCT-18GE" split air co